# Counting people and drug exposures in an OMOP database with SQL

**SYNTHETIC DATA.** The Eunomia GiBleed database holds simulated patients, not real people. The Book of OHDSI calls it a simulated dataset in the CDM. Use it to learn the tables and the SQL. Its counts say nothing about celecoxib or bleeding in real patients.

**Data kit lab** `omop-sql-counts-eunomia` · Real-world · about 30 minutes

**The question.** How do you count people, people exposed to a drug, and diagnoses during exposure in an OMOP common data model (CDM) database with standard SQL?

**The data.** GiBleed, OMOP CDM 5.3, one CSV per table, from the OHDSI EunomiaDatasets repository (Apache License 2.0).

**The published result.** The Book of OHDSI, Appendix E Suggested Answers: Exercises 9.1 to 9.3 (section E.5) and 5.1 (section E.2).

**What you do.**

1. Open the zip and load each table into an in-memory SQLite database.
2. Read the book's own queries from its answers page and run them unchanged.
3. Count persons, persons exposed to celecoxib (two ways), and gastrointestinal hemorrhage diagnoses during celecoxib exposure.
4. Look up a standard concept id in the vocabulary tables.
5. Compare the five numbers with the book.

In [ ]:
# datakit:setup (generated from lab.yaml and expected.json by scripts/datakit/harness.py sync; do not edit by hand)
import hashlib, json, os, pathlib, urllib.request

LAB = 'omop-sql-counts-eunomia'
FILES = json.loads(r'''{
 "gibleed": {
  "url": "https://raw.githubusercontent.com/OHDSI/EunomiaDatasets/main/datasets/GiBleed/GiBleed_5.3.zip",
  "name": "gibleed.zip",
  "sha256": "081b82e84c90e2904e02c26546e46412b01cce566071c77e0c4a355c2d0612f9",
  "bytes": 6861852
 },
 "book-queries": {
  "url": "https://ohdsi.github.io/TheBookOfOhdsi/SuggestedAnswers.html",
  "name": "book-queries.html",
  "sha256": "42544f0b749ce7b69b82f82fdc5615e6b8e2441c6d35a44f8d09d9f6f57b301f",
  "bytes": 171081
 }
}''')
PUBLISHED = json.loads(r'''{
 "person_count": {
  "label": "Persons in the PERSON table",
  "value": 2694,
  "tolerance": {
   "exact": true
  }
 },
 "celecoxib_persons_drug_exposure": {
  "label": "Persons with a celecoxib exposure (DRUG_EXPOSURE via CONCEPT_ANCESTOR)",
  "value": 1844,
  "tolerance": {
   "exact": true
  }
 },
 "celecoxib_persons_drug_era": {
  "label": "Persons with a celecoxib drug era (DRUG_ERA)",
  "value": 1844,
  "tolerance": {
   "exact": true
  }
 },
 "gi_bleed_diagnoses_during_celecoxib": {
  "label": "Diagnoses of gastrointestinal hemorrhage (192671 and descendants) inside a celecoxib drug era",
  "value": 41,
  "tolerance": {
   "exact": true
  }
 },
 "gi_hemorrhage_concept_id": {
  "label": "Standard concept id of Gastrointestinal hemorrhage",
  "value": 192671,
  "tolerance": {
   "exact": true
  }
 }
}''')
PUBLISHED_FROM = 'The Book of OHDSI, Appendix E Suggested Answers (HTML, no page numbers): section E.5 SQL and R, Exercises 9.1, 9.2 and 9.3; section E.2 Standardized Vocabularies, Exercise 5.1 | https://ohdsi.github.io/TheBookOfOhdsi/SuggestedAnswers.html'
DATA_DIR = pathlib.Path(os.environ.get('DATAKIT_DATA', 'datakit-data/' + LAB))


def data_file(file_id):
    """Path of one dataset file. Downloads it from its source on first use and checks the checksum."""
    f = FILES[file_id]
    path = DATA_DIR / f['name']
    if not path.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print('downloading', f['url'])
        urllib.request.urlretrieve(f['url'], path)
    if f['sha256']:
        got = hashlib.sha256(path.read_bytes()).hexdigest()
        if got != f['sha256']:
            print('WARNING:', path.name, 'differs from the copy this lab was checked against (the source file changed)')
    return path


def within(computed, check):
    tol, pub = check['tolerance'], check['value']
    if 'exact' in tol:
        return computed == pub
    limit = tol['abs'] if 'abs' in tol else tol['rel'] * abs(pub)
    return abs(computed - pub) <= limit + 1e-12


def report(results):
    """Print computed against published values and save the computed values for the harness."""
    print('Published values:', PUBLISHED_FROM)
    print(f"{'check':<44}{'published':>12}{'computed':>14}  result")
    for key, check in PUBLISHED.items():
        got = results.get(key)
        shown = '(missing)' if got is None else (f'{got:.4f}' if isinstance(got, float) else str(got))
        verdict = 'missing' if got is None else ('match' if within(got, check) else 'DIFFERENT')
        print(f"{check['label'][:43]:<44}{str(check['value']):>12}{shown:>14}  {verdict}")
    out = pathlib.Path(os.environ.get('DATAKIT_RESULTS', 'results.json'))
    out.write_text(json.dumps(results, indent=1))
    print('saved', out)


## 1. Open the zip

The zip holds one CSV per CDM table. It also holds `__MACOSX/` entries: resource files that macOS adds to a zip. They are not tables; skip them.

In [ ]:
import html
import re
import sqlite3
import zipfile

import pandas as pd

zf = zipfile.ZipFile(data_file("gibleed"))
members = [n for n in zf.namelist() if n.endswith(".csv") and not n.startswith("__MACOSX/")]
print(len(zf.namelist()), "entries in the zip,", len(members), "CSV tables")

## 2. Load the tables into SQLite

SQLite comes with Python. Its default schema is called `main`. The book's queries are written with a placeholder `@cdm` for the schema, and the book fills it with `main`. So the book's SQL runs here once `@cdm.` becomes `main.`.

The CSV headers are upper case (`PERSON_ID`). SQLite does not care about the case of table and column names.

In [ ]:
con = sqlite3.connect(":memory:")
sizes = {}
for name in members:
    table = name.rsplit("/", 1)[-1][:-4].lower()
    frame = pd.read_csv(zf.open(name), low_memory=False)
    frame.to_sql(table, con, index=False)
    sizes[table] = len(frame)
print(pd.Series(sizes).sort_values(ascending=False).head(12).to_string())

for table, column in [("drug_era", "DRUG_ERA_START_DATE"), ("drug_era", "DRUG_ERA_END_DATE"), ("condition_occurrence", "CONDITION_START_DATE")]:
    bad = con.execute(f"SELECT COUNT(*) FROM {table} WHERE {column} NOT GLOB '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]'").fetchone()[0]
    print(f"{table}.{column}: {bad} values not in YYYY-MM-DD form")

SQLite stores these dates as text. Text comparison orders ISO dates (`YYYY-MM-DD`) correctly, which is why the check above matters: the book compares dates with `>=` and `<=`.

## 3. The book's queries, read from its answers page

We do not retype the SQL. We read it from the Suggested Answers page (file `book-queries`), section E.5, and print it. Only the query text is taken from the page.

In [ ]:
page = data_file("book-queries").read_text(encoding="utf-8")
parts = re.split(r'<h2><span class="header-section-number">(E\.\d+)</span>', page)
sections = {parts[i]: " ".join(html.unescape(re.sub(r"(?s)<[^>]+>", " ", parts[i + 1])).split()) for i in range(1, len(parts), 2)}
book_sql = re.findall(r'sql <- "(.*?)"', sections["E.5"])
assert len(book_sql) == 4, len(book_sql)
for label, sql in zip(["9.1", "9.2 (DRUG_EXPOSURE)", "9.2 (DRUG_ERA)", "9.3"], book_sql):
    print(f"Exercise {label}:\n  {sql}\n")


def run(sql):
    return con.execute(sql.replace("@cdm.", "main.")).fetchone()[0]

## 4. Exercise 9.1: how many people?

One row per person in `PERSON`.

In [ ]:
RESULTS = {}
RESULTS["person_count"] = run(book_sql[0])
print("persons:", RESULTS["person_count"])

## 5. Exercise 9.2: how many people have at least one prescription of celecoxib?

Two routes.

- `DRUG_EXPOSURE`: to find all drugs containing the ingredient celecoxib, the book joins through `CONCEPT_ANCESTOR` to the ingredient concept.
- `DRUG_ERA` is already rolled up to the ingredient, so a join to `CONCEPT` is enough.

Both count `DISTINCT person_id`: one person can have many exposures.

In [ ]:
RESULTS["celecoxib_persons_drug_exposure"] = run(book_sql[1])
RESULTS["celecoxib_persons_drug_era"] = run(book_sql[2])
print("persons, via DRUG_EXPOSURE:", RESULTS["celecoxib_persons_drug_exposure"])
print("persons, via DRUG_ERA:     ", RESULTS["celecoxib_persons_drug_era"])
rows = con.execute(book_sql[1].replace("@cdm.", "main.").replace("COUNT(DISTINCT(person_id))", "COUNT(*)")).fetchone()[0]
print("exposure rows behind them:  ", rows)

## 6. Exercise 9.3: diagnoses of gastrointestinal hemorrhage during celecoxib exposure

The exercise asks how many diagnoses of gastrointestinal hemorrhage (concept 192671 and everything below it in the hierarchy) start while a person is exposed to celecoxib. The book uses `DRUG_ERA`, because eras of one ingredient do not overlap and so a diagnosis is not counted twice.

First, the query exactly as the book prints it.

In [ ]:
printed = run(book_sql[3])
RESULTS["gi_bleed_diagnoses_during_celecoxib"] = printed
print("book's query as printed (the checked value):", printed)

That count is the value this lab checks against the book (owner's decision of 11 Nov 2026, SPEC.txt amendment 1).

Now read the query's join to `CONDITION_OCCURRENCE`: it matches on dates only. Nothing says the diagnosis and the drug era belong to the same person. "During exposure" is about one person, so a query that answers the exercise needs `condition_occurrence.person_id = drug_era.person_id` as well. Run that version and compare. Its count is shown, not checked.

In [ ]:
join_on_dates = "ON condition_start_date >= drug_era_start_date"
assert book_sql[3].count(join_on_dates) == 1
same_person_sql = book_sql[3].replace(join_on_dates, "ON condition_occurrence.person_id = drug_era.person_id AND condition_start_date >= drug_era_start_date")
print(same_person_sql, "\n")
person_matched = run(same_person_sql)
print("person-matched count (shown, not checked):", person_matched)

pairs = con.execute(book_sql[3].replace("@cdm.", "main.").replace(
    "SELECT COUNT(*) AS diagnose_count",
    "SELECT COUNT(*), SUM(drug_era.person_id = condition_occurrence.person_id)")).fetchone()
print(f"pairs found by the printed query: {pairs[0]}, of which the same person: {pairs[1]}")

**The printed query reproduces the book's number, but it does not answer the exercise.** The book prints 41 for Exercise 9.3, and its printed query gives the count shown above on this file. Look at the last line of the output: it says how many of those pairs join a diagnosis and a drug era of the same person. With the person match the count is the one printed above. `report` below compares the printed query's count with the book.

Why is the person-matched count what it is? Look at the people who have both a celecoxib era and a bleeding diagnosis, and where the diagnosis falls relative to the era.

In [ ]:
timing = pd.read_sql("""
WITH era AS (
  SELECT drug_era.person_id AS pid, MIN(drug_era_start_date) AS era_start, MAX(drug_era_end_date) AS era_end
  FROM main.drug_era INNER JOIN main.concept ingredient ON drug_concept_id = ingredient.concept_id
  WHERE LOWER(ingredient.concept_name) = 'celecoxib' AND ingredient.concept_class_id = 'Ingredient' AND ingredient.standard_concept = 'S'
  GROUP BY drug_era.person_id),
bleed AS (
  SELECT person_id AS pid, condition_start_date AS d
  FROM main.condition_occurrence INNER JOIN main.concept_ancestor ON condition_concept_id = descendant_concept_id
  WHERE ancestor_concept_id = 192671)
SELECT COUNT(DISTINCT bleed.pid) AS persons_with_both,
       SUM(bleed.d < era.era_start) AS before_era,
       SUM(bleed.d BETWEEN era.era_start AND era.era_end) AS inside_era,
       SUM(bleed.d > era.era_end) AS after_era
FROM bleed INNER JOIN era ON bleed.pid = era.pid""", con)
timing

## 7. Exercise 5.1: the standard concept id of gastrointestinal hemorrhage

The book answers this one with the ATHENA vocabulary browser. The GiBleed database carries a small subset of the vocabulary, enough to answer it with SQL.

In [ ]:
hits = pd.read_sql("""SELECT concept_id, concept_name, domain_id, vocabulary_id, standard_concept
FROM main.concept WHERE concept_name = 'Gastrointestinal hemorrhage' AND standard_concept = 'S'""", con)
print(hits.to_string(index=False))
assert len(hits) == 1
# SQLite returns column names as they were loaded: upper case, from the CSV headers
RESULTS["gi_hemorrhage_concept_id"] = int(hits["CONCEPT_ID"].iloc[0])

print("\nvocabulary subset in this database:")
print(pd.read_sql("SELECT vocabulary_id, COUNT(*) AS concepts FROM main.concept GROUP BY vocabulary_id ORDER BY concepts DESC", con).to_string(index=False))

Exercises 5.2 and 5.3 ask for the ICD-10-CM, ICD-9-CM and MedDRA codes that map to this concept. The subset above shows which of those vocabularies GiBleed carries; they are not checked here.

## 8. Compare with the book

`PUBLISHED` holds the book's five values, copied by code from the answers page, not typed. All are integers and must match exactly.

In [ ]:
RESULTS = {k: int(v) for k, v in RESULTS.items()}
report(RESULTS)

## What to take away

- Persons, rows and events are different counts. Say which one you report.
- Ingredient-level questions need the vocabulary hierarchy (`CONCEPT_ANCESTOR`) or the era tables.
- A join on dates alone pairs one person's diagnosis with another person's drug era. Section 6 shows how many of the printed query's pairs belong to the same person.
- A printed answer can come from a query with an error. Reproducing the number is not the same as checking the query.

## Try next

1. Count celecoxib *exposure rows* per person. What is the largest number?
2. Run Exercise 9.3 against `DRUG_EXPOSURE` instead of `DRUG_ERA`, with the person match. Does the count change?
3. Change the window to "within 30 days after the era ends". How many diagnoses now?
4. Repeat Exercise 9.2 for diclofenac.
5. Open the OMOP CDM documentation and find which table holds the observation periods. Restrict Exercise 9.1 to people observed in 2010.